# Solutions — Notebook 03: Data Warehouse

**BINFX410 — Chapter 10**

> **Prerequisite:** Run `03_data_warehouse.ipynb` first (which requires `01_introduction_and_setup.ipynb`). This builds the `../warehouse/genomics.duckdb` database with all star-schema tables.

In [ ]:
import duckdb

con = duckdb.connect('../warehouse/genomics.duckdb')
print(f'Connected to genomics.duckdb')
for (tbl,) in con.execute('SHOW TABLES').fetchall():
    cnt = con.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f'  {tbl:<25} {cnt:>6,} rows')

## Exercise 3.1 — Cohort Analysis

A cohort analysis groups samples by the month their first variant call was processed, then tracks how many samples from each cohort had variant calls in subsequent months.

Write a DuckDB query that produces a cohort table with:
- `cohort_year`, `cohort_month` — when the sample first had a PASS call
- `months_since_first_call` — 0, 1, 2, ...
- `samples_active` — how many samples from that cohort had a call in that relative month
- `call_count` — total PASS calls from that cohort in that relative month

In [ ]:
cohort = con.execute("""
    WITH first_calls AS (
        -- Find the date of each sample's first PASS variant call
        SELECT
            fvc.sample_key,
            MIN(dd.full_date) AS first_call_date
        FROM fact_variant_calls fvc
        JOIN dim_date dd ON fvc.date_key = dd.date_key
        WHERE fvc.filter_status = 'PASS'
        GROUP BY fvc.sample_key
    ),
    cohorts AS (
        SELECT
            sample_key,
            first_call_date,
            YEAR(first_call_date)  AS cohort_year,
            MONTH(first_call_date) AS cohort_month
        FROM first_calls
    )
    SELECT
        c.cohort_year,
        c.cohort_month,
        -- months since first call for each subsequent call
        DATEDIFF('month', c.first_call_date, dd.full_date) AS months_since_first_call,
        COUNT(DISTINCT fvc.sample_key)                      AS samples_active,
        COUNT(DISTINCT fvc.call_id)                         AS call_count
    FROM fact_variant_calls fvc
    JOIN dim_date dd ON fvc.date_key = dd.date_key
    JOIN cohorts  c  ON fvc.sample_key = c.sample_key
    WHERE fvc.filter_status = 'PASS'
      AND dd.full_date >= c.first_call_date
    GROUP BY c.cohort_year, c.cohort_month, months_since_first_call
    ORDER BY c.cohort_year, c.cohort_month, months_since_first_call
""").df()

print('=== Sample Cohort Analysis ===')
print(cohort.head(20).to_string(index=False))

## Exercise 3.2 — Running Total and Moving Average

Write a query that produces a monthly time series of PASS variant counts with:
- `year`, `month`
- `monthly_variants`
- `cumulative_variants` — running total of variants from the start
- `rolling_3mo_avg` — 3-month moving average of monthly variant counts

Use DuckDB window functions (`SUM() OVER`, `AVG() OVER` with a ROWS frame).

In [ ]:
rolling = con.execute("""
    WITH monthly AS (
        SELECT
            dd.year,
            dd.month,
            COUNT(fv.variant_key)  AS monthly_variants
        FROM fact_variants fv
        JOIN fact_variant_calls fvc ON fv.call_key = fvc.call_key
        JOIN dim_date            dd ON fv.date_key  = dd.date_key
        WHERE fvc.filter_status = 'PASS'
        GROUP BY dd.year, dd.month
    )
    SELECT
        year,
        month,
        monthly_variants,
        SUM(monthly_variants) OVER (
            ORDER BY year, month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_variants,
        ROUND(AVG(CAST(monthly_variants AS DOUBLE)) OVER (
            ORDER BY year, month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 1) AS rolling_3mo_avg
    FROM monthly
    ORDER BY year, month
""").df()

print('=== Monthly Variant Counts with Running Total and 3-Month Moving Average ===')
print(rolling.to_string(index=False))

## Exercise 3.3 — Gene Co-occurrence Analysis

Which pairs of genes are most frequently mutated in the **same variant call** (same `call_id`)? This is analogous to a market basket analysis — genes that co-occur may be in the same pathway.

Write a query that returns the top 10 gene pairs (by co-occurrence count). Each row should have:
- `gene_a`, `gene_b`
- `co_occurrence_count`

Hint: Self-join `fact_variants` on `call_id` where `gene_key_a < gene_key_b` to avoid duplicate pairs.

In [ ]:
cooccurrence = con.execute("""
    SELECT
        dg1.gene_name  AS gene_a,
        dg2.gene_name  AS gene_b,
        dg1.biotype    AS biotype_a,
        dg2.biotype    AS biotype_b,
        COUNT(*)       AS co_occurrence_count
    FROM fact_variants fv1
    JOIN fact_variants fv2
      ON  fv1.call_id    = fv2.call_id      -- same variant call
      AND fv1.gene_key  < fv2.gene_key      -- avoid duplicates and self-pairs
    JOIN dim_genes dg1 ON fv1.gene_key = dg1.gene_key
    JOIN dim_genes dg2 ON fv2.gene_key = dg2.gene_key
    JOIN fact_variant_calls fvc ON fv1.call_key = fvc.call_key
    WHERE fvc.filter_status = 'PASS'
    GROUP BY dg1.gene_name, dg2.gene_name, dg1.biotype, dg2.biotype
    ORDER BY co_occurrence_count DESC
    LIMIT 10
""").df()

print('=== Top 10 Gene Co-occurrence Pairs (PASS calls only) ===')
print(cooccurrence.to_string(index=False))

con.close()